In [12]:
%load_ext autoreload
%autoreload 2

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
assert os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] == "false"

import sys
sys.path.append('../../')

import ATLAS

import json, pickle
from glob import glob
from tqdm import tqdm

import numpy as np
from matplotlib import pyplot as plt

from enterprise.pulsar import Pulsar

# Make Pint shut up
import pint.logging
import logging
pint.logging.setup(level=logging.WARNING)
logging.getLogger('enterprise').setLevel(logging.WARNING)

ideal_path = '../../datasets/ideal_NG15'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
parfiles, timfiles = get_partim(ideal_path)

import pint.config
import pint.fitter
from pint.models import get_model, get_model_and_toas
from pint.residuals import Residuals
from pint.simulation import make_fake_toas_uniform

m, t = get_model_and_toas(parfiles[0], timfiles[0])

In [33]:
import pint.fitter 
f = pint.fitter.Fitter.auto(t, m)
f.fit_toas()
st = f.get_derived_params()


WARNING  (pint.logging                  ): /home/kyleg/miniforge3/envs/atlas/lib/python3.12/site-packages/pint/models/timing_model.py:3525 FutureWarning: AffineScalarFunc.__lt__() is deprecated. It will be removed in a future release.


In [34]:
print(st)

Derived Parameters:
Period = 0.0053621004674176548±0.0000000000000000005 s
Pdot = (1.7839991±0.0000006)×10⁻²⁰
Characteristic age = 4.762e+09 yr (braking index = 3)
Surface magnetic field = 3.13e+08 G
Magnetic field at light cylinder = 1.869e+04 G
Spindown Edot = 4.568e+33 erg / s (I=1e+45 cm2 g)

Parallax distance = 1341.2±57.0 pc

Binary model BinaryELL1
Conversion from ELL1 parameters:
ECC = (2.1688±0.0005)×10⁻⁵
OM  = 276.458±0.009 deg
T0  = 56220.55449(32)
Checking applicability of ELL1 model -- 
    Condition is asini/c * ecc**4 << timing precision / sqrt(# TOAs) to use ELL1
    asini/c * ecc**4    = 2.04e-12 us 
    TRES / sqrt(# TOAs) = 6.03e-06 us 
    Should be fine.

Mass function = 0.00555739257(8) Msun
Min / Median Companion mass (assuming Mpsr = 1.4 Msun) = 0.2470 / 0.2902 Msun
From SINI in model:
    cos(i) = 0.0462(12)
    i = 87.35(7) deg
Pulsar mass (Shapiro Delay) = 1.5316112427452497 solMass


In [ ]:
def get_name(parOrTim):
    return parOrTim.split('/')[-1].split('_')[0]

def get_partim(path):
    parfiles = sorted(glob(f'{path}/par/*.par'))
    timfiles = sorted(glob(f'{path}/tim/*.tim'))
    assert len(parfiles) == len(timfiles)
    for p,t in zip(parfiles, timfiles):
        assert get_name(p) == get_name(t), f'{p, t}'
    return parfiles, timfiles

def parse_par(parfile):
    with open(parfile, 'r') as f:
        lines = f.readlines()

    # Parameters are either:
    # 1. Name  value
    # 2. Name  value [1] error
    # 3. Name -flags value
    # 3. Name -flags value [1] error

    par_dict = {}
    for i in range(len(lines)):
        line = lines[i].strip() 
        if line.startswith('#'):
            continue

        line = line.split()
        if '1' in line:
            # This is a fit parameter
            fidx = line.index('1') # Element before fit flag is value, after is error
            param_name = ' '.join(line[:fidx-1])
            vals = (float(line[fidx-1]), float(line[fidx+1]))
            par_dict[param_name] = vals
        else:
            # Non fit parameter
            param_names = ' '.join(line[:-1])
            try:
                vals = (float(line[-1]), None) # Try interpreting as float
            except:
                vals = (line[-1], None) # But if it doesn't work, use string
            par_dict[param_names] = vals

    return par_dict

from scipy.stats import norm, uniform, truncnorm
binary_priors = {
    'BINARY':   None,       # Binary timing model (DD, DDK, BT, ELL1, ELL1H)
    'PB':       None,       # Orbital period (days)
    'PBDOT':    None,       # Time derivative of orbital period
    'A1':       None,       # Projected semi-major axis (light-seconds)
    'A1DOT':    None,       # Time derivative of A1
    'T0':       None,       # Epoch of periastron passage
    'TASC':     None,       # Epoch of ascending node passage
    'ECC':      None,       # Orbital eccentricity
    'EDOT':     None,       # Time derivative of eccentricity
    'OM':       None,       # Longitude of periastron (deg)
    'OMDOT':    None,       # Time derivative of OM
    'EPS1':     None,       # Laplace-Lagrange parameter e*sin(omega)
    'EPS2':     None,       # Laplace-Lagrange parameter e*cos(omega)
    'SINI':     None,       # sin(inclination)
    'M2':       None,       # Companion mass (solar masses)
    'KIN':      None,       # Inclination angle (deg), used by DDK
    'KOM':      None,       # Longitude of ascending node (deg), used by DDK
    'GAMMA':    None,       # Einstein delay parameter
    'A0':       None,       # Aberration parameter A0
    'B0':       None,       # Aberration parameter B0
    'DR':       None,       # Damour-Deruelle parameter dr
    'DTH':      None,       # Damour-Deruelle parameter dtheta
    'H3':       None,       # Orthometric Shapiro delay amplitude
    'NHARMS':   None,       # Number of harmonics (ELL1H)
    'FB0':      None,       # Orbital frequency and derivatives (ELL1)
    'FB1':      None,     
    'FB2':      None,
    'FB3':      None,
    'FB4':      None,
    'FB5':      None,
}


def get_binary_params(par_dict):
    return {k: v for k, v in par_dict.items() if k in binary_priors.keys()}




In [4]:
par, tim = get_partim(ideal_path)
par_dicts = [parse_par(p) for p in par]
npsrs = len(par_dicts)

binary_par_dicts = [d for d in par_dicts if 'BINARY' in d]
nbinary = len(binary_par_dicts)

print(f'{npsrs=}, {nbinary=}')

# How many unique models do we have?
binary_models = set(d['BINARY'][0] for d in binary_par_dicts)
print(f'All binary models={binary_models}')


npsrs=67, nbinary=49
All binary models={'DDK', 'BT', 'ELL1', 'ELL1H', 'DD'}


In [11]:
for par in binary_priors.keys():
    if par == 'BINARY':
        continue

    # Get all values of this parameter in all pulsars (if exists)
    vals = []
    for pardict in binary_par_dicts:
        if par in pardict:
            vals.append(pardict[par])

    vals = np.array(vals)
    print(par,vals)

    


PB [[1.23271712e+01 1.15371550e-10]
 [1.17349097e+02 1.52472718e-08]
 [6.95571721e+00 3.79825131e-09]
 [5.74104238e+00 1.81483767e-10]
 [4.90797689e+00 5.79015437e-09]
 [4.84655044e+00 2.79651400e-09]
 [5.56723340e+01 1.99373302e-07]
 [2.86016002e-01 1.31936111e-10]
 [1.19851256e+00 5.76920313e-12]
 [4.36667627e+00 5.14127689e-10]
 [4.76694462e+00 8.80427300e-11]
 [6.04672714e-01 4.84060716e-12]
 [3.79724632e+01 4.25953057e-08]
 [7.80513016e+00 4.11823422e-10]
 [1.53554460e+01 1.39939296e-09]
 [3.85038328e+01 2.83784036e-08]
 [7.61745675e+01 4.75399883e-09]
 [1.43484631e+01 1.28499366e-06]
 [8.68661942e+00 3.42685901e-11]
 [1.25246233e+01 1.13082399e-04]
 [1.75460662e+02 3.09527009e-09]
 [1.47017396e+02 1.19474770e-08]
 [6.78251299e+01 7.63479550e-10]
 [9.07062816e-02 6.65783097e-10]
 [3.54790734e-01 6.31201439e-12]
 [1.63353478e+01 2.23200049e-10]
 [7.30241421e-01 1.01285581e-09]
 [1.10746459e+02 1.22081712e-07]
 [6.98889243e-01 2.25268116e-11]
 [6.27230197e+00 4.04048198e-10]
 [1.156

In [9]:
from ATLAS.utils import T_SUN_SEC

def can_use_method_1(par_dict):
    required_keys = ['PB', 'A1', 'M2', 'SINI']
    return all(key in par_dict for key in required_keys)

def method_1(par_dict):
    PB = par_dict['PB'][0] * 86400  # Convert days to seconds
    A1 = par_dict['A1'][0] # Already in seconds (light seconds)
    M2 = par_dict['M2'][0] # Companion mass in solar masses
    SINI = par_dict['SINI'][0] # Sine of inclination angle
    ECC = par_dict['ECC'][0] if 'ECC' in par_dict else 0

    # Use keplerian mass function relation
    Mtot2 = (M2*SINI)**3 * (T_SUN_SEC*PB**2)/(4*np.pi**2 * A1**3) * (1 - ECC**2)**(3/2)
    Mtot = np.sqrt(Mtot2)
    M1 = Mtot - M2

    return M1


In [12]:
for pdict in binary_par_dicts:
    if can_use_method_1(pdict):
        M1 = method_1(pdict)
        print(pdict['PSR'][0],'\t', M1)
    else:
        pass
        #print(pdict['PSR'][0], "Cannot use methods")






B1855+09 	 1.5317843463201923
J0613-0200 	 2.9690136891852594
J0740+6620 	 2.0015939413234167
J1600-3053 	 1.4274712767147295
J1614-2230 	 1.934380911801516
J1630+3734 	 6.253322238395491
J1640+2224 	 2.7871351059707243
J1741+1351 	 0.9757751534687872
J1811-2405 	 2.1411116945248887
J1853+1303 	 0.6062991667493648
J1903+0327 	 0.9367932109711387
J1909-3744 	 1.492539071236712
J1918-0642 	 1.2418395365626291
J1946+3417 	 7.013443297916851
J2017+0603 	 2.0196044795142787
J2043+1711 	 1.5880769348285138
J2302+4442 	 1.1242382740694072
